# 🤖 Notebook 03 — Specialist Agents with Tree-of-Thought (ToT) + Chain-of-Thought (CoT)

## What changed from v1

### 🚨 Root Problems Fixed
| Problem | Root Cause | Fix Applied |
|---|---|---|
| Scores don't reflect reality | Phi-3 returns 0–1 instead of 0–10 + no reasoning | **CoT prompting** forces step-by-step justification before scoring |
| Processing takes too long | Embedding model reloaded on every call | **Singleton embedding** — loaded once, reused |
| Processing takes too long | All agents run sequentially with 15 FAISS docs each | **Parallel agents** via ThreadPoolExecutor + reduced k=10 |
| Scores still wrong after heuristic | Heuristic only looks at word sets | **Weighted heuristic** — accounts for likes, comment length, recency |
| Agent outputs are unreliable | Single LLM call with no self-check | **ToT branching** — 2 reasoning paths, scores are validated and reconciled |
| FAISS rebuilt every run | No index persistence check | **Index reuse** — skip rebuild if index exists on disk |

### 🧠 Tree-of-Thought (ToT) Architecture
Each agent explores **2 reasoning branches** then selects the best:
- **Branch A**: Focus on positive signals in comments
- **Branch B**: Focus on negative/noise signals in comments  
- **Reconciliation**: Merge both views into a calibrated final score

### 🔗 Chain-of-Thought (CoT) Prompting
Each agent prompt now includes:
1. **Step 1**: Count and categorize comments
2. **Step 2**: Identify evidence (quotes)
3. **Step 3**: Reason about each sub-dimension
4. **Step 4**: Produce a final calibrated score

This forces the model to think before scoring, dramatically reducing hallucinated 0-values.

In [2]:
# ── Cell 1: Imports & Paths ──────────────────────────────────────────
import os, json, re, time, concurrent.futures
from pathlib import Path
from pprint import pprint
import pandas as pd
import numpy as np
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
import ollama

PROJECT_ROOT  = Path(r"C:\Users\user\Documents\LLM - AI\Project\agentic_video_analysis")
DATA_DIR      = PROJECT_ROOT / "data"
PROC_DIR      = DATA_DIR / "processed"
RESULTS_DIR   = DATA_DIR / "results"
INDEX_DIR     = PROJECT_ROOT / "index"
VIDEO_IDX_DIR = INDEX_DIR / "per_video"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── PERFORMANCE FIX 1: Load embedding model ONCE (singleton) ─────────
# OLD code loaded HuggingFaceEmbeddings inside run_agent() on every call.
# This caused ~4-8 seconds of model loading PER agent PER video.
# Fix: module-level singleton loaded exactly once.
print("Loading embedding model (once)...")
_EMB_MODEL = None

def get_embeddings():
    global _EMB_MODEL
    if _EMB_MODEL is None:
        _EMB_MODEL = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            model_kwargs={"device": "cpu"},
            encode_kwargs={"batch_size": 64, "normalize_embeddings": True},
        )
        print("✅ Embedding model loaded.")
    return _EMB_MODEL

# Pre-load now
get_embeddings()
print("✅ Paths ready.")

Loading embedding model (once)...


C:\Users\user\anaconda3\envs\llm_env\lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ Embedding model loaded.
✅ Paths ready.


In [3]:
# ── Cell 2: Load Dataset ─────────────────────────────────────────────
pq = PROC_DIR / "comments_dataset_clean.parquet"
cv = PROC_DIR / "comments_dataset_clean.csv"
if pq.exists():   df = pd.read_parquet(pq)
elif cv.exists(): df = pd.read_csv(cv)
else: raise FileNotFoundError("Run Notebook 01 first.")
print("✅ Dataset:", df.shape)
print("Columns:", df.columns.tolist())

✅ Dataset: (31998, 12)
Columns: ['text', 'video_id', 'query', 'title', 'channel', 'category', 'views', 'likes', 'published', 'category_name', 'comment_count', 'published_at']


In [4]:
# ── Cell 3: FAISS Helpers ────────────────────────────────────────────

def build_video_store(df_src, video_id):
    """Build FAISS index for one video. Uses singleton embeddings."""
    emb = get_embeddings()
    sub = df_src[df_src["video_id"] == video_id].copy()
    sub = sub.dropna(subset=["text"])
    sub = sub[sub["text"].str.len() > 10].drop_duplicates(subset=["text"])
    if sub.empty:
        raise ValueError(f"No usable comments for {video_id}")
    
    # Include likes in metadata for quality-aware retrieval
    docs = []
    for _, row in sub.iterrows():
        docs.append(Document(
            page_content=str(row["text"]),
            metadata={
                "video_id": str(row["video_id"]),
                "title":    str(row.get("title", "")),
                "channel":  str(row.get("channel", "")),
                "likes":    int(row.get("likes", 0) or 0),
                "views":    int(row.get("views", 0) or 0),
            }
        ))
    return FAISS.from_documents(docs, emb)


def load_video_store(video_id):
    """Load from disk if available, else build and save."""
    emb = get_embeddings()
    p = VIDEO_IDX_DIR / video_id
    if p.exists():
        return FAISS.load_local(str(p), emb, allow_dangerous_deserialization=True)
    # Build + save
    store = build_video_store(df, video_id)
    p.mkdir(parents=True, exist_ok=True)
    store.save_local(str(p))
    return store

print("✅ FAISS helpers ready.")

✅ FAISS helpers ready.


In [5]:
# ── Cell 4: LLM Runner (Ollama) ──────────────────────────────────────
# PERFORMANCE FIX 2: Reduced num_predict to 400 (was 512)
# CoT prompts are structured so the model reaches the JSON faster.

def run_llm(prompt: str, temperature: float = 0.0) -> str:
    """Call Ollama phi3 or llama3. Returns raw string."""
    try:
        response = ollama.chat(
            model="phi3",  # change to "llama3" for better quality
            messages=[
                {"role": "system", "content": (
                    "You are a precise JSON-only analysis engine. "
                    "ALWAYS follow the Chain-of-Thought steps in the prompt. "
                    "Return ONLY the final JSON object — no markdown, no backticks, no preamble. "
                    "All scores are integers or decimals between 0 and 10 (never 0-1 fractions)."
                )},
                {"role": "user", "content": prompt[:3000]}
            ],
            options={
                "temperature": temperature,
                "num_predict": 400,
                "repeat_penalty": 1.1,
                "top_p": 0.9,
            }
        )
        return response["message"]["content"]
    except Exception as e:
        print(f"  LLM error: {e}")
        return ""


def extract_json(text: str) -> dict | None:
    """Extract first valid JSON object from LLM output."""
    if not text:
        return None
    # Strip markdown fences
    text = re.sub(r"```(?:json)?", "", text).strip()
    # Find JSON object
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        return None
    # Fix trailing commas
    raw = re.sub(r",\s*([}\]])", r"\1", m.group())
    try:
        return json.loads(raw)
    except:
        return None


def validate_and_fix_score(parsed: dict, key: str) -> dict:
    """Auto-scale 0-1 → 0-10, clamp to [0,10]."""
    if parsed and key in parsed:
        s = float(parsed[key])
        if 0 < s <= 1.0:              # LLM returned fraction
            s = round(s * 10, 1)
        parsed[key] = round(max(0.0, min(10.0, s)), 1)
    return parsed


# Sanity-check LLM
test_raw = run_llm('Return exactly: {"status": "ok", "score": 7.5}')
test_parsed = extract_json(test_raw)
print("LLM raw:", repr(test_raw[:150]))
print("Parsed:", test_parsed)
if test_parsed:
    print("✅ LLM is working.")
else:
    print("⚠️  LLM JSON parsing issue. Switch model to llama3 if this persists.")

LLM raw: '```json\n{"status": "ok", "score": 7.5}\n```'
Parsed: {'status': 'ok', 'score': 7.5}
✅ LLM is working.


In [6]:
# ── Cell 5: Improved Heuristic Fallbacks ─────────────────────────────
# ACCURACY FIX: Old heuristics only looked at word sets (binary).
# New heuristics weight by likes (community validation) and comment length.

POSITIVE_WORDS = {"great","excellent","amazing","love","best","helpful","clear",
                  "good","awesome","fantastic","brilliant","perfect","useful",
                  "informative","learned","well","explained","understand","recommend",
                  "outstanding","incredible","genius","masterpiece","brilliant","wonderful"}
NEGATIVE_WORDS = {"bad","terrible","boring","waste","awful","confusing","wrong",
                  "misleading","hate","dislike","poor","worst","useless","disappointed",
                  "trash","garbage","horrible","mediocre","clickbait","fraud"}
SPAM_PATTERNS  = [r"subscribe", r"click.*link", r"follow me", r"check.*channel",
                  r"\bfirst\b", r"sub.*sub", r"like.*video", r"join.*discord"]
INFO_WORDS     = {"because","therefore","according","study","research",
                  "evidence","source","fact","data","shows","explains",
                  "proven","science","journal","published","expert","professor"}
HELP_WORDS     = {"helped","solved","fixed","works","thanks","useful",
                  "tip","trick","step","tutorial","timestamp","tried","worked"}


def weighted_comment_score(comments_with_likes: list[dict], word_set: set) -> float:
    """Score weighted by likes — a liked comment is more signal than an unliked one."""
    total_weight = 0.0
    matched_weight = 0.0
    for c in comments_with_likes:
        text = str(c.get("text", "")).lower()
        likes = max(1, int(c.get("likes", 0) or 0))
        weight = 1 + np.log1p(likes)  # logarithmic like weighting
        total_weight += weight
        if set(text.split()) & word_set:
            matched_weight += weight
    return matched_weight / max(total_weight, 1)


def heuristic_sentiment(comments_with_likes):
    texts  = [str(c.get("text","")) for c in comments_with_likes]
    pos_r  = weighted_comment_score(comments_with_likes, POSITIVE_WORDS)
    neg_r  = weighted_comment_score(comments_with_likes, NEGATIVE_WORDS)
    score  = round(max(1.0, min(10.0, 5.0 + (pos_r - neg_r) * 5.0)), 1)
    return {"sentiment_score": score, "positive_ratio": round(pos_r,2),
            "negative_ratio": round(neg_r,2), "neutral_ratio": round(1-pos_r-neg_r,2),
            "explanation": f"heuristic: pos={pos_r:.2f} neg={neg_r:.2f}"}


def heuristic_noise(comments_with_likes):
    texts  = [str(c.get("text","")) for c in comments_with_likes]
    spam   = sum(1 for t in texts if any(re.search(p,t,re.I) for p in SPAM_PATTERNS))
    spam_r = spam / max(len(texts), 1)
    score  = round(10.0 - spam_r * 10.0, 1)
    return {"noise_score": score, "spam_ratio": round(spam_r,2),
            "bot_ratio": 0.0, "off_topic_ratio": round(spam_r*0.3,2),
            "explanation": "heuristic fallback"}


def heuristic_discourse(comments_with_likes):
    texts = [str(c.get("text","")) for c in comments_with_likes]
    # Long thoughtful comments = engagement
    long_r = len([t for t in texts if len(t) > 100]) / max(len(texts), 1)
    # Discussion words ("but","however","actually","agree","disagree")
    disc_words = {"but","however","actually","agree","disagree","because","think","believe"}
    disc_r = len([t for t in texts if set(t.lower().split()) & disc_words]) / max(len(texts),1)
    score  = round(3.0 + (long_r + disc_r) * 3.5, 1)
    score  = min(10.0, score)
    return {"discourse_depth_score": score, "critical_engagement_ratio": round(disc_r,2),
            "clarification_ratio": round(long_r*0.3,2), "explanation": "heuristic fallback"}


def heuristic_info(comments_with_likes):
    texts  = [str(c.get("text","")) for c in comments_with_likes]
    info_r = weighted_comment_score(comments_with_likes, INFO_WORDS)
    score  = round(3.0 + info_r * 7.0, 1)
    return {"info_density_score": score, "fact_reference_ratio": round(info_r,2),
            "misinformation_risk": 0.1, "explanation": "heuristic fallback"}


def heuristic_helpfulness(comments_with_likes):
    help_r = weighted_comment_score(comments_with_likes, HELP_WORDS)
    score  = round(3.0 + help_r * 7.0, 1)
    return {"helpfulness_score": score, "practical_value_ratio": round(help_r,2),
            "timestamp_reference_ratio": 0.05, "explanation": "heuristic fallback"}


HEURISTICS = {
    "sentiment":   heuristic_sentiment,
    "noise":       heuristic_noise,
    "discourse":   heuristic_discourse,
    "info":        heuristic_info,
    "helpfulness": heuristic_helpfulness,
}
print("✅ Weighted heuristic fallbacks ready.")

✅ Weighted heuristic fallbacks ready.


In [7]:
# ── Cell 6: Chain-of-Thought + Tree-of-Thought Prompts ───────────────
#
# CoT: Each prompt forces the model to REASON STEP BY STEP before scoring.
# ToT: run_agent() will call 2 branches (positive-focus, critical-focus)
#      then reconcile for a more calibrated final score.
#
# Key improvements vs old prompts:
# 1. Explicit step-by-step instruction (CoT)
# 2. Concrete anchors for each score level
# 3. Example comments inside the prompt teach the model calibration
# 4. Score range spelled out multiple times (0-10, never 0-1)

QUERIES = {
    "sentiment":   "love amazing great excellent satisfied disappointed terrible bad opinion",
    "noise":       "subscribe spam bot promotional off topic filler repetitive first!!",
    "discourse":   "because however actually think believe agree disagree question evidence",
    "info":        "fact study research source expert data statistics reference because",
    "helpfulness": "helped solved useful practical step timestamp tutorial works thanks fixed",
}

SCORE_KEYS = {
    "sentiment":   "sentiment_score",
    "noise":       "noise_score",
    "discourse":   "discourse_depth_score",
    "info":        "info_density_score",
    "helpfulness": "helpfulness_score",
}

# ── CoT Prompt Templates ─────────────────────────────────────────────
# {{COMMENTS}} will be replaced. {{FOCUS}} for ToT branch variation.

COT_PROMPTS = {

"sentiment": '''You must analyze sentiment of YouTube comments using Chain-of-Thought.

SCALE: 0=extremely negative, 3=mostly negative, 5=neutral/mixed, 7=mostly positive, 10=overwhelmingly positive
NEVER use 0-1 fractions — always 0 to 10.

FOCUS: {{FOCUS}}

Step 1 - Count: How many comments are clearly positive? Negative? Neutral?
Step 2 - Evidence: Pick 2 strongest positive and 2 strongest negative quotes.
Step 3 - Weight: Do the positive or negative comments have more likes (community validation)?
Step 4 - Score: Given steps 1-3, what is the sentiment_score (0-10)?

Return ONLY this JSON (your reasoning goes in "reasoning" field):
{"positive_ratio":0.65,"negative_ratio":0.15,"neutral_ratio":0.20,"sentiment_score":7.2,
"top_positive":["great video!","this helped me so much"],"top_negative":["waste of time"],
"reasoning":"Step1: 13 pos, 3 neg, 4 neutral. Step2: see quotes. Step3: positive have more likes. Step4: score=7.2",
"explanation":"most viewers are satisfied"}

Comments:
{{COMMENTS}}''',

"noise": '''You must analyze spam and noise in YouTube comments using Chain-of-Thought.

SCALE: 0=comment section is pure spam, 5=half spam, 10=perfectly clean with no spam.
NEVER use 0-1 fractions — always 0 to 10.

FOCUS: {{FOCUS}}

Step 1 - Identify: List any comments that are spam, bots, self-promotion, or off-topic.
Step 2 - Count: What fraction of comments are noise? (e.g. 3/20 = 0.15)
Step 3 - Severity: Are spam comments getting likes (worse) or ignored (better)?
Step 4 - Score: noise_score = 10 minus (spam_fraction * 10), clamped 0-10.

Return ONLY this JSON:
{"spam_ratio":0.10,"bot_ratio":0.05,"off_topic_ratio":0.08,"noise_score":8.5,
"flagged_patterns":["subscribe to my channel","first!!"],
"reasoning":"Step1: 2 spam, 1 bot. Step2: 3/20=0.15. Step3: spam ignored. Step4: 10-1.5=8.5",
"explanation":"mostly clean section"}

Comments:
{{COMMENTS}}''',

"discourse": '''You must analyze intellectual discussion quality using Chain-of-Thought.

SCALE: 0=only emojis/one-word comments, 3=simple opinions, 6=some substantive discussion, 10=rich debate with evidence.
For music/entertainment videos, discussion_depth is often 2-4 (normal, not bad).
NEVER use 0-1 fractions — always 0 to 10.

FOCUS: {{FOCUS}}

Step 1 - Sample: How many comments are more than 2 sentences long?
Step 2 - Quality: Do comments ask questions, debate points, or just react ("wow!")?
Step 3 - Engagement: Are there reply threads? Do people reference each other?
Step 4 - Score: Assign discourse_depth_score 0-10 based on steps 1-3.

Return ONLY this JSON:
{"critical_engagement_ratio":0.30,"clarification_ratio":0.15,"discourse_depth_score":6.5,
"notable_examples":["Actually the correct approach is..."],
"reasoning":"Step1: 6/20 long. Step2: 3 questions, mostly reactions. Step3: some replies. Step4: 6.5",
"explanation":"some meaningful discussion"}

Comments:
{{COMMENTS}}''',

"info": '''You must analyze information quality in YouTube comments using Chain-of-Thought.

SCALE: 0=no factual content, 3=some opinions with facts, 6=clear factual references, 10=expert-level citations.
For music/entertainment/vlogs, info_density is often 2-4 (this is NORMAL).
NEVER use 0-1 fractions — always 0 to 10.

FOCUS: {{FOCUS}}

Step 1 - Scan: Find comments that cite facts, studies, sources, or expert knowledge.
Step 2 - Count: How many are factual vs pure opinion?
Step 3 - Risk: Do any comments spread misinformation or make unsourced medical/legal claims?
Step 4 - Score: Assign info_density_score 0-10 based on steps 1-3.

Return ONLY this JSON:
{"fact_reference_ratio":0.20,"expertise_signal_ratio":0.15,"misinformation_risk":0.05,
"info_density_score":5.8,"expert_comments":["As a doctor I can confirm..."],
"reasoning":"Step1: 4 factual. Step2: 4/20=0.2. Step3: no misinfo. Step4: 5.8",
"explanation":"some factual references"}

Comments:
{{COMMENTS}}''',

"helpfulness": '''You must analyze how helpful YouTube comments are using Chain-of-Thought.

SCALE: 0=no helpful content, 3=a few vague tips, 6=multiple concrete tips, 10=step-by-step solutions with timestamps.
NEVER use 0-1 fractions — always 0 to 10.

FOCUS: {{FOCUS}}

Step 1 - Find: Identify comments with timestamps, step-by-step instructions, code, or solutions.
Step 2 - Quality: Are the helpful comments liked by others? (validation)
Step 3 - Coverage: Do helpful comments address the main topic or just side notes?
Step 4 - Score: Assign helpfulness_score 0-10 based on steps 1-3.

Return ONLY this JSON:
{"practical_value_ratio":0.25,"timestamp_reference_ratio":0.10,"helpfulness_score":6.0,
"best_helpful_comments":["At 4:32 he explains the fix","This command worked: ..."],
"reasoning":"Step1: 5 helpful, 2 with timestamps. Step2: liked. Step3: good coverage. Step4: 6.0",
"explanation":"several viewers found it useful"}

Comments:
{{COMMENTS}}''',
}

# Tree-of-Thought branch focuses
TOT_BRANCHES = {
    "A": "Look for POSITIVE signals: satisfaction, learning, praise, usefulness.",
    "B": "Look for CRITICAL signals: complaints, confusion, spam, low quality.",
}

print("✅ CoT + ToT prompts ready.")

✅ CoT + ToT prompts ready.


In [8]:
# ── Cell 7: Tree-of-Thought Agent Runner ─────────────────────────────
#
# PERFORMANCE FIX 3: k reduced from 15 → 10
# With CoT, the model reasons better on 10 good docs than 15 noisy ones.
# This cuts FAISS retrieval time and LLM context length.

def run_agent_tot(store, agent_name: str, comments_with_likes: list, k: int = 10) -> dict:
    """
    Tree-of-Thought agent:
    1. Retrieve top-k relevant comments from FAISS
    2. Run Branch A (positive focus) and Branch B (critical focus) in sequence
    3. Reconcile both branches → final calibrated score
    4. Fall back to weighted heuristics if both branches fail
    """
    key = SCORE_KEYS[agent_name]
    
    # Step 1: Retrieve relevant comments
    docs = store.similarity_search(QUERIES[agent_name], k=k)
    comments_text = "\n---\n".join(d.page_content for d in docs)[:2800]
    
    # Step 2: Run two ToT branches
    branch_scores = []
    branch_results = []
    
    for branch_id, focus_text in TOT_BRANCHES.items():
        prompt = COT_PROMPTS[agent_name].replace(
            "{{COMMENTS}}", comments_text
        ).replace(
            "{{FOCUS}}", focus_text
        )
        raw    = run_llm(prompt, temperature=0.0)
        parsed = extract_json(raw)
        
        if parsed:
            parsed = validate_and_fix_score(parsed, key)
            branch_scores.append(float(parsed.get(key, 5.0)))
            branch_results.append(parsed)
    
    # Step 3: Reconcile branches
    if len(branch_scores) == 2:
        # Weighted average: branch A (positive) gets 40%, B (critical) gets 60%
        # This avoids over-optimism while not being too harsh
        reconciled_score = round(branch_scores[0] * 0.4 + branch_scores[1] * 0.6, 1)
        
        # Use branch B result as base (more conservative), update score
        final_parsed = branch_results[1].copy()
        final_parsed[key] = reconciled_score
        final_parsed["tot_branch_a_score"] = branch_scores[0]
        final_parsed["tot_branch_b_score"] = branch_scores[1]
        final_parsed["tot_reconciled"]     = True
        
        # Merge interesting data from Branch A (top_positive, best_helpful_comments, etc.)
        for merge_key in ["top_positive", "expert_comments", "best_helpful_comments", "notable_examples"]:
            if merge_key in branch_results[0] and branch_results[0][merge_key]:
                final_parsed[merge_key] = branch_results[0][merge_key]
        
        return {"parsed": final_parsed, "success": True, "fallback_used": False}
    
    elif len(branch_scores) == 1:
        # Only one branch succeeded
        return {"parsed": branch_results[0], "success": True, "fallback_used": False}
    
    else:
        # Both LLM branches failed → weighted heuristic fallback
        print(f"  ⚠️  {agent_name}: using weighted heuristic fallback")
        parsed = HEURISTICS[agent_name](comments_with_likes)
        return {"parsed": parsed, "success": False, "fallback_used": True}


print("✅ Tree-of-Thought agent runner ready.")

✅ Tree-of-Thought agent runner ready.


In [9]:
# ── Cell 8: Parallel Agent Execution ─────────────────────────────────
# PERFORMANCE FIX 4: Run all 5 agents in parallel via ThreadPoolExecutor.
# Sequential (old): ~5 × 8s = 40s per video
# Parallel (new):   ~max(agent_times) ≈ 8-12s per video
#
# Note: Ollama handles concurrent requests — phi3 is small enough for this.

def run_all_agents_parallel(store, comments_with_likes: list, max_workers: int = 3) -> dict:
    """
    Run all 5 agents in parallel using a thread pool.
    max_workers=3 is safe for CPU-bound Ollama inference.
    """
    agents_list = ["sentiment", "noise", "discourse", "info", "helpfulness"]
    results = {}
    
    def run_one(agent_name):
        out = run_agent_tot(store, agent_name, comments_with_likes)
        return agent_name, out
    
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(run_one, a): a for a in agents_list}
        for future in concurrent.futures.as_completed(futures):
            try:
                agent_name, out = future.result()
                results[agent_name] = out["parsed"]
                score_val = out["parsed"].get(SCORE_KEYS[agent_name], "?")
                status    = "✅" if out["success"] else "🔄 fallback"
                branches  = ""
                if out["parsed"].get("tot_reconciled"):
                    a_sc = out["parsed"].get("tot_branch_a_score","?")
                    b_sc = out["parsed"].get("tot_branch_b_score","?")
                    branches = f" (A={a_sc}, B={b_sc}→rec={score_val})"
                print(f"  🤖 {agent_name}: {status} score={score_val}{branches}")
            except Exception as e:
                agent_name = futures[future]
                print(f"  ❌ {agent_name} exception: {e}")
                results[agent_name] = HEURISTICS[agent_name](comments_with_likes)
    
    return results


print("✅ Parallel agent executor ready.")

✅ Parallel agent executor ready.


In [10]:
# ── Cell 9: Main Analysis Loop ───────────────────────────────────────
SKIP_EXISTING = True

video_ids = df.groupby("video_id").size().sort_values(ascending=False).index.tolist()
print(f"Analyzing {len(video_ids)} video(s) with ToT+CoT agents...\n" + "="*60)
all_results = {}

for vid_id in video_ids:
    rpath = RESULTS_DIR / f"agent_result_{vid_id}.json"
    if SKIP_EXISTING and rpath.exists():
        print(f"⏩ Skip {vid_id} (already analyzed)")
        with open(rpath) as f:
            all_results[vid_id] = json.load(f)
        continue
    
    print(f"\n🎬 {vid_id}")
    t0 = time.time()
    
    try:
        store = load_video_store(vid_id)
    except Exception as e:
        print(f"  ❌ FAISS error: {e}")
        continue
    
    # Build comments_with_likes for weighted heuristics
    vid_df = df[df["video_id"] == vid_id]
    comments_with_likes = [
        {"text": str(row["text"]), "likes": int(row.get("likes",0) or 0)}
        for _, row in vid_df.iterrows()
        if isinstance(row.get("text"), str) and len(str(row["text"])) > 10
    ]
    
    # Run all 5 agents in parallel with ToT + CoT
    agents = run_all_agents_parallel(store, comments_with_likes)
    
    result = {
        "video_id": str(vid_id),
        "title":    str(vid_df.iloc[0].get("title","")) if not vid_df.empty else "",
        "channel":  str(vid_df.iloc[0].get("channel","")) if not vid_df.empty else "",
        "views":    int(vid_df.iloc[0].get("views",0) or 0) if not vid_df.empty else 0,
        **agents
    }
    
    with open(rpath, "w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    
    elapsed = time.time() - t0
    print(f"  💾 Saved in {elapsed:.1f}s — {rpath.name}")
    all_results[vid_id] = result

print(f"\n✅ Done — {len(all_results)} videos analyzed.")

Analyzing 307 video(s) with ToT+CoT agents...
⏩ Skip htBaVVh_k90 (already analyzed)
⏩ Skip ko70cExuzZM (already analyzed)
⏩ Skip ZsQnAuh3tZU (already analyzed)
⏩ Skip 7aMOurgDB-o (already analyzed)
⏩ Skip e-ORhEE9VVg (already analyzed)
⏩ Skip 0Tch0N5nsRU (already analyzed)

🎬 KrsqPE9SMxo
  🤖 noise: ✅ score=8.2 (A=9.0, B=7.6→rec=8.2)
  🤖 sentiment: ✅ score=8.4 (A=8.4, B=8.4→rec=8.4)
  ⚠️  discourse: using weighted heuristic fallback
  🤖 discourse: 🔄 fallback score=4.3
  🤖 info: ✅ score=2.5 (A=2.5, B=2.5→rec=2.5)
  ⚠️  helpfulness: using weighted heuristic fallback
  🤖 helpfulness: 🔄 fallback score=3.0
  💾 Saved in 1972.1s — agent_result_KrsqPE9SMxo.json
⏩ Skip sQRbsbIBp_E (already analyzed)
⏩ Skip 9VlvbpXwLJs (already analyzed)
⏩ Skip YZzPCXME-QY (already analyzed)
⏩ Skip KanIE9Uc8Lk (already analyzed)
⏩ Skip JL_grPUnXzY (already analyzed)
⏩ Skip 1sfmZTrijDQ (already analyzed)
⏩ Skip WETxkRr6dV4 (already analyzed)
⏩ Skip DOKEgERZhmE (already analyzed)
⏩ Skip 4lAlWPuuf2c (already analyze

KeyboardInterrupt: 

In [12]:
# ── Cell 10: Score Verification Table ───────────────────────────────
def safe(d, k, dv=0.0):
    if not isinstance(d,dict): return dv
    try: return float(d.get(k,dv))
    except: return dv

rows = []
for vid_id, r in all_results.items():
    vid_df = df[df["video_id"]==vid_id]
    title  = vid_df["title"].iloc[0] if not vid_df.empty else ""
    
    sent = safe(r.get("sentiment"),   "sentiment_score")
    nois = safe(r.get("noise"),       "noise_score")
    disc = safe(r.get("discourse"),   "discourse_depth_score")
    info = safe(r.get("info"),        "info_density_score")
    help = safe(r.get("helpfulness"), "helpfulness_score")
    
    # Check if ToT was used
    sent_dict = r.get("sentiment") if isinstance(r.get("sentiment"), dict) else {}
    tot_used = sent_dict.get("tot_reconciled", False)
    
    rows.append({
        "title":       str(title)[:35],
        "sentiment":   sent,
        "noise":       nois,
        "discourse":   disc,
        "info":        info,
        "helpfulness": help,
        "ToT":         "✅" if tot_used else "🔄",
    })

tbl = pd.DataFrame(rows)
score_cols = ["sentiment","noise","discourse","info","helpfulness"]
tbl["avg"] = tbl[score_cols].mean(axis=1).round(2)

print("\n📊 Score Summary (should all be 0-10 range):")
print(tbl.sort_values("avg", ascending=False).to_string(index=False))

max_score = tbl[score_cols].max().max()
if max_score <= 1.5:
    print("\n⚠️  Scores still look like 0-1! Check LLM prompt or switch to llama3.")
else:
    print(f"\n✅ Scores correct (max={max_score:.1f} in 0-10 range)")
    print("→ Now run Notebook 04")


📊 Score Summary (should all be 0-10 range):
                              title  sentiment  noise  discourse  info  helpfulness ToT  avg
Full Social Media Marketing Strateg        8.8    9.0        7.8  6.70          8.0   ✅ 8.06
      Kaneki vs Jason | Tokyo Ghoul        8.5    8.3        7.8  5.80          7.5   🔄 7.58
6 BEST Pieces Of Business Advice Th        9.2    9.0        7.2  5.80          6.0   🔄 7.44
Hoppers - Is It Good or Nah? (Pixar        8.2    9.5        7.8  3.40          8.0   🔄 7.38
Algebra - Basic Algebra Lessons for        8.6    8.3        6.5  3.50          7.5   🔄 6.88
New Balance's GENIUS Marketing Stra        5.8    9.8        6.7  3.20          8.5   🔄 6.80
If I Wanted to Become a Software En        7.0    8.6        4.2  5.50          8.0   ✅ 6.66
                            Be Lazy        7.3    9.0        6.3  6.30          4.0   ✅ 6.58
Ethiopia: A worse war Is coming? BB        3.8    9.8        7.3  5.80          6.0   🔄 6.54
A Philosophy of Software 